<a href="https://colab.research.google.com/github/ksw9179/AI_and_Data-/blob/main/%EA%B0%9C%EC%84%A0%EC%95%88_%EC%95%94_%ED%99%98%EC%9E%90_%EB%B9%85%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_%EB%B0%8F_%EC%98%88%ED%9B%84_%EC%98%88%EC%B8%A1_ML%5BTop%5D_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Stage 1: 암 환자 빅데이터 분석 및 예후 예측 ML
**목표:** TCGA(The Cancer Genome Atlas) 형태의 암 환자 유전체 데이터(RNA-seq)와 임상 데이터(Clinical)를 병합하고, 머신러닝(Random Forest/XGBoost)을 통해 환자의 생존 여부를 예측하는 핵심 타겟 유전자를 발굴.

## 💡 핵심 하위 개념 (Bottom-up)
1. **TCGA 데이터 구조:** 암 환자의 임상 정보(나이, 병기, 생존 여부 등)와 분자 데이터(유전자 발현량 등)가 고유 환자 ID(Barcode)로 연결된 대규모 데이터베이스.
2. **RNA-seq 정규화 (TPM vs FPKM):** - 유전자 발현량을 잴 때, 유전자 길이가 길거나 시퀀싱을 많이 할수록 숫자가 커지는 왜곡을 방지하기 위한 정규화 방법.
   - 최근에는 샘플 간 비교가 더 직관적인 **TPM(Transcripts Per Million)**을 표준으로 많이 사용.
3. **불균형 데이터(Imbalanced Data):** 의료 데이터 특성상 '생존' 환자가 '사망' 환자보다 훨씬 많음. 이를 보정하기 위해 알고리즘에 가중치(`class_weight`)를 줍니다.

In [ ]:
# 필수 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
# XGBoost가 설치되어 있다면 활성화하세요! (pip install xgboost)
from xgboost import XGBClassifier

# 그래프 스타일 설정
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

# ==========================================
# 1. TCGA 모의 데이터(Mock Data) 생성
# 실전에서는 UCSC Xena 웹사이트 등에서 csv를 다운로드하여 pd.read_csv()로 불러옵니다.
# ==========================================
np.random.seed(42)
n_samples = 500
n_genes = 100

# 1) Clinical 데이터 생성 (환자 ID, 나이, 생존 여부)
# 생존(Alive, 0)이 80%, 사망(Dead, 1)이 20%인 불균형 데이터 가정
patient_ids = [f"TCGA-A1-{i:04d}" for i in range(n_samples)]
survival_status = np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
clinical_df = pd.DataFrame({
    'Patient_ID': patient_ids,
    'Age': np.random.randint(30, 80, size=n_samples),
    'Target_Survival': survival_status # 0: Alive, 1: Dead
})

# 2) RNA-seq 데이터 생성 (TPM 정규화 발현량 가정)
# 사망 환자(1)에게 특정 유전자(예: Gene_42, Gene_7)의 발현량이 눈에 띄게 높도록 인위적 조작 (바이오마커 역할)
gene_names = [f"Gene_{i}" for i in range(n_genes)]
rna_data = np.random.lognormal(mean=2, sigma=1, size=(n_samples, n_genes))
rna_df = pd.DataFrame(rna_data, columns=gene_names)
rna_df['Patient_ID'] = patient_ids

# 특정 유전자를 진짜 바이오마커처럼 작동하게 만듦







# 진짜 진짜 핵심 부분. Step 2 의 Dead 예측을 못하는 게으른 AI 예방하기 위한 부분

# ==========================================
# [수정된 바이오마커 주입 방식]
# 단순 덧셈이 아니라, 사망 환자의 특정 유전자 스케일 자체를 곱하기 배수로 증폭시켜
# 스케일러 통과 후에도 AI가 확실한 시그널을 잡을 수 있도록 만듭니다.
# ==========================================

# rna_df.loc[clinical_df['Target_Survival'] == 1, 'Gene_42'] += 15.0  <-- 기존 코드 주석 처리
# rna_df.loc[clinical_df['Target_Survival'] == 1, 'Gene_7'] += 8.0    <-- 기존 코드 주석 처리

# 곱하기 배수로 변경 (사망 환자군에서 발현량이 확실히 높게 뜀)
rna_df.loc[clinical_df['Target_Survival'] == 1, 'Gene_42'] = 150.0   # 그래도 안되니 =150.0 기존은 *5.0
rna_df.loc[clinical_df['Target_Survival'] == 1, 'Gene_7'] = 80.0   # 역시 =80.0 기존은 *3.0

print("✅ 바이오마커 시그널 증폭이 완료되었습니다.")

print("✅ TCGA 형태의 모의 데이터 생성이 완료되었습니다.")
print(f"- Clinical 데이터 크기: {clinical_df.shape}")
print(f"- RNA-seq 데이터 크기: {rna_df.shape}")

In [ ]:
print(clinical_df[:10])

In [ ]:
print(rna_df[:10])

## 🛠️ Step 1. 데이터 병합 및 전처리
임상 데이터와 RNA-seq 데이터를 `Patient_ID`를 기준으로 병합(Merge)합니다.
분석을 위해 유전자 발현량(Feature, `X`)과 생존 여부(Target, `y`)를 분리합니다.

In [ ]:
# ==========================================
# [수정된 2번 전처리 셀 코드]
# StandardScaler 대신 생물학 빅데이터의 표준인 Log 변환 후
# 0과 1 사이로 압축하는 MinMaxScaler를 조합.
# ==========================================
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 병합 (Inner Join)
merged_df = pd.merge(clinical_df, rna_df, on='Patient_ID')

# 2. 입력 데이터(X)와 타겟 데이터(y) 분리
# 분석에 불필요한 환자 ID, 나이 등은 제외하고 오직 '유전자 발현량'만 X로 사용.
X = merged_df.drop(columns=['Patient_ID', 'Age', 'Target_Survival'])
y = merged_df['Target_Survival']

# 3. 생물학적 로그 변환 (데이터 분포의 왜곡을 막고 차이를 보존)
# 데이터에 0이 있을 수 있으므로 안전하게 log(x + 1) 처리를 합니다.
X_log = np.log1p(X)

# 4. Train / Test 데이터 분할 (8:2 비율)
# stratify 옵션의 중요성(특성 불균형 비율 보존)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)  # stratify=y로 생존,사망 특성 비율 보존이 핵심!

# 데이터 스케일링 (표준화)
# 유전자마다 발현량 편차가 커서 평균 0, 분산 1로 맞춤, 이때 fit_transform(기준) vs transform(새로운 데이터는 기준x)
scaler = MinMaxScaler()  # 가장 중요한 개선 포인트 = 게으른 AI 방지
X_train_scaled = scaler.fit_transform(X_train)   # train 데이터는 평균,분산 계산한 뒤 변환(기준)
X_test_scaled = scaler.transform(X_test)  # test 데이터는 train 데이터를 기준으로 변환

# 6. 컬럼명 복원을 위해 다시 DataFrame으로 변환
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("✅ 시그널 손실 없는 새로운 전처리가 완료되었습니다.")
print(f"학습용 데이터 크기: {X_train_scaled.shape}")
print(f"검증용 데이터 크기: {X_test_scaled.shape}")

## 🤖 Step 2. 머신러닝 모델 학습 (Random Forest)
생존 여부를 예측하는 분류 모델을 만듭니다.
데이터가 불균형하므로(생존자가 훨씬 많음), 모델이 다수 클래스에 편향되지 않도록 `class_weight='balanced'` 옵션을 사용하여 소수 클래스(사망자)의 오답에 더 큰 패널티를 부여합니다.

In [ ]:
# Random Forest 모델 생성 및 학습
# n_estimators: 사용할 의사결정 나무의 개수
# class_weight='balanced': 불균형 데이터 해결을 위한 핵심 옵션
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)

# 검증 데이터로 예측 수행
y_pred = rf_model.predict(X_test_scaled)

# 모델 성능 평가
print("=== 암 환자 생존 예측 분류기 성능 평가 ===")
print(classification_report(y_test, y_pred, target_names=['Alive (0)', 'Dead (1)']))

# Confusion Matrix 시각화
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Alive', 'Dead'], yticklabels=['Alive', 'Dead'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 🎯 Step 3. 최상위 타겟 바이오마커 유전자 추출
AI가 환자의 생존 여부를 가를 때 **가장 중요하게 생각한 유전자(Feature Importance)** 순위를 매기고, 1등 유전자를 발굴합니다.

In [ ]:
# Feature Importance 추출 및 데이터프레임화
feature_importances = pd.DataFrame({
    'Gene': X.columns,
    'Importance': rf_model.feature_importances_
})

# 중요도 순으로 내림차순 정렬
top_genes = feature_importances.sort_values(by='Importance', ascending=False)

# 최상위 타겟 유전자 1개 선정
top_1_gene = top_genes.iloc[0]['Gene']
top_1_score = top_genes.iloc[0]['Importance']

print(f"🏆 AI가 발굴한 최상위 타겟 유전자: {top_1_gene} (중요도 점수: {top_1_score:.4f})")

# Top 10 유전자 시각화
plt.figure(figsize=(10, 6))
sns.barplot(data=top_genes.head(10), x='Importance', y='Gene', palette='Reds_r')
plt.title('Top 10 Biomarker Genes for Survival Prediction')
plt.xlabel('Random Forest Feature Importance')
plt.ylabel('Genes')
plt.show()

# 1등 유전자의 실제 발현량 차이 시각화 (Boxplot)
plt.figure(figsize=(6, 5))
sns.boxplot(data=merged_df, x='Target_Survival', y=top_1_gene, palette='Set1')
plt.title(f'Expression Level of Top Gene ({top_1_gene}) by Survival Status')
plt.xticks([0, 1], ['Alive', 'Dead'])
plt.ylabel(f'{top_1_gene} Expression (TPM)')
plt.show()